# Underwater Waste Detection — Saved YOLO Model (fixed Kaggle paths)

Load your saved checkpoint, automatically locate underwater dataset images under Kaggle's input folders, generate annotated predictions, and download them. **No training is run.**

Before running, attach:
- Model: `mohamedselimoueslati/underwater`
- Original underwater dataset, under Add Input → Datasets

The notebook searches `/kaggle/input/datasets` first, then other `/kaggle/input` folders. It prints the exact selected paths. Verify that the model is the final fine-tuned checkpoint before attributing predictions to that model.


In [ ]:
%pip -q install ultralytics

## 1. Settings

In [ ]:
from pathlib import Path
import random

MODEL_PATH = None  # Set the .pt file path if more than one model is attached
DATASET_DIR = Path("/kaggle/input/datasets")  # Parent folder that Kaggle uses on your session
IMAGE_DIR = None  # Optional: exact path to the folder with sample images

NUM_IMAGES = 8
CONFIDENCE = 0.25
IMAGE_SIZE = 640
SEED = 42
OUTPUT_DIR = Path("/kaggle/working/underwater_predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Find the saved model

In [ ]:
from ultralytics import YOLO

if MODEL_PATH:
    weights = Path(MODEL_PATH)
else:
    model_candidates = sorted(
        (p for p in Path("/kaggle/input").rglob("*.pt")
         if "underwater" in str(p).lower()),
        key=lambda p: (p.name != "best.pt", len(str(p)))
    )
    print("Underwater checkpoint candidates:")
    for p in model_candidates:
        print(" -", p)
    if not model_candidates:
        raise FileNotFoundError(
            "No underwater .pt file found. Use Add Input > Models to attach "
            "mohamedselimoueslati/underwater, then rerun this cell."
        )
    if len(model_candidates) != 1:
        raise ValueError(
            "More than one checkpoint found. Copy the correct path into "
            "MODEL_PATH in the Settings cell and rerun."
        )
    weights = model_candidates[0]

if not weights.is_file():
    raise FileNotFoundError(f"Missing model file: {weights}")

print("Loading:", weights)
model = YOLO(str(weights))
print("Task:", model.task)
print("Classes:", model.names)
if model.task != "detect":
    raise ValueError("This notebook expects a YOLO object-detection checkpoint.")

if isinstance(getattr(model, "ckpt", None), dict):
    print("Checkpoint epoch (if recorded):", model.ckpt.get("epoch", "unknown"))

print("Check that this is your FINAL trained checkpoint, not the starting checkpoint.")

## 3. Locate images

In [ ]:
from pathlib import Path

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

# Kaggle may mount datasets at /kaggle/input/datasets/<owner>/<dataset>/...
# Other notebooks use /kaggle/input/<dataset>/...
def sample_images(folder):
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in EXTENSIONS)


def folder_rank(folder):
    parts = [x.lower() for x in folder.parts]
    for name, score in [("test", 0), ("valid", 1), ("val", 1), ("validation", 1), ("train", 2)]:
        if name in parts:
            return score
    return 3


if IMAGE_DIR:
    image_folder = Path(IMAGE_DIR)
    if not image_folder.is_dir():
        raise FileNotFoundError(f"IMAGE_DIR does not exist: {image_folder}")
    images = sample_images(image_folder)
else:
    image_folder = None
    images = []
    roots = [DATASET_DIR, Path("/kaggle/input/underwater-plastic-pollution-detection"), Path("/kaggle/input")]
    for root in roots:
        if not root.is_dir():
            continue
        # Prefer folders named 'images', where YOLO datasets normally store examples.
        folders = [p for p in root.rglob("*") if p.is_dir() and p.name.lower() == "images"]
        folders.sort(key=lambda p: ("underwater" not in str(p).lower(), folder_rank(p), len(p.parts)))
        for candidate in folders:
            candidate_images = sample_images(candidate)
            if candidate_images:
                image_folder, images = candidate, candidate_images
                break
        if image_folder is not None:
            break

    # Fallback for datasets where files are directly within a folder, without /images.
    if image_folder is None:
        for root in roots:
            if not root.is_dir():
                continue
            matches = sample_images(root)
            if matches:
                preferred = [p for p in matches if "underwater" in str(p).lower()]
                matches = preferred or matches
                images = matches
                image_folder = root
                break

if not images:
    print("Available folders under /kaggle/input:")
    for folder in Path("/kaggle/input").glob("*"):
        print(" -", folder)
    raise FileNotFoundError("No JPG/PNG/WEBP files found. Attach the underwater image DATASET, not only the model, or provide IMAGE_DIR.")

selected = random.Random(SEED).sample(images, min(NUM_IMAGES, len(images)))
print("Image source:", image_folder)
print("Available images:", len(images))
print("Selected samples:")
for path in selected:
    print(" -", path)


## 4. Run inference and save annotated images

In [ ]:
import csv
from IPython.display import display, Image

summary = []
for i, image_path in enumerate(selected, start=1):
    result = model.predict(
        source=str(image_path),
        imgsz=IMAGE_SIZE,
        conf=CONFIDENCE,
        verbose=False
    )[0]

    filename = OUTPUT_DIR / f"prediction_{i:02d}.jpg"
    result.save(filename=str(filename))
    count = 0 if result.boxes is None else len(result.boxes)
    summary.append((image_path.name, count, filename.name))

    print(f"{filename.name}: {count} detections | source: {image_path.name}")
    display(Image(filename=str(filename), width=700))

with (OUTPUT_DIR / "predictions.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["input_image", "detections", "output_image"])
    writer.writerows(summary)

print("Images saved to:", OUTPUT_DIR)

## 5. Download all pictures

In [ ]:
import shutil
from IPython.display import FileLink, display

zip_path = Path("/kaggle/working/underwater_predictions.zip")
shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=OUTPUT_DIR)
print("ZIP:", zip_path)
display(FileLink(str(zip_path)))

## Optional: Generate new validation figures

This is **not necessary** for prediction screenshots. If your dataset includes the correct `data.yaml` and labels, set `RUN_VALIDATION=True` to create new evaluation plots. These are *2026 evaluation outputs*, not original 2025 figures.

In [ ]:
RUN_VALIDATION = False

if RUN_VALIDATION:
    candidates = list(DATASET_DIR.rglob("data.yaml")) if DATASET_DIR.is_dir() else []
    if not candidates:
        candidates = list(Path("/kaggle/input").rglob("data.yaml"))
    if not candidates:
        raise FileNotFoundError("No data.yaml found in attached dataset; provide the correct dataset and labels")
    if len(candidates) > 1:
        print("Found multiple YAMLs. Choose the correct one:")
        for path in candidates:
            print(" -", path)
        raise ValueError("Set yaml_path manually to the correct data.yaml")
    yaml_path = candidates[0]
    print("Validation YAML:", yaml_path)
    metrics = model.val(
        data=str(yaml_path),
        split="val",
        imgsz=IMAGE_SIZE,
        plots=True,
        project="/kaggle/working",
        name="underwater_validation",
        exist_ok=True,
    )
    print("mAP@50:", metrics.box.map50)
    print("mAP@50-95:", metrics.box.map)
    print("Plots: /kaggle/working/underwater_validation/")


## What to use in your README

Select a few representative `prediction_XX.jpg` files (including mistakes, if present). Label them **“Inference examples generated in November 2026.”**

Before describing them as predictions from your **final 2025 fine-tuned model**, verify that the checkpoint you attached is the saved output of the completed 100-epoch run, not the earlier checkpoint used to initialize it. Also check the dataset's permission for redistribution of example images.